# TIDE-GR on Amazon Reviews 2023 — Video_Games (5-core) (Colab T4)

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

How resuming works:
* Every run saves a checkpoint after each block (and every 5 warm-up epochs) into the **shared Drive folder**.
* If the session ends or the GPU quota runs out, open this notebook on **any** account that has the shared
  folder added to "My Drive" (right-click the shared folder → *Organize → Add shortcut to My Drive*), and
  run all cells again. Finished runs are skipped, the unfinished run continues from its last checkpoint.
* Progress: `<RUNS_ROOT>/amazon_vg/status.json`.

The first run downloads the 5-core ratings (~13 MB) and the item metadata, and encodes item text once (cached in the shared folder).

In [ ]:
!nvidia-smi -L

## 1. Settings — edit `RUNS_ROOT` to the name of your shared folder in *My Drive*

In [ ]:
RUNS_ROOT = "/content/drive/MyDrive/TIDE_GR_runs"   # <- shared folder (same path on every account)
REPO = "https://github.com/S-V-Kartheek/TIDE_GR.git"
MAX_HOURS = 11.0    # stop cleanly before the Colab limit; re-run to continue
SHEET = "TIDE-GR results"   # <- Google Sheet for all metrics: paste its URL (best, shared by all accounts) or a name
SA_KEY = None               # optional: path to a service-account .json key if the Google login cell fails

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(RUNS_ROOT, exist_ok=True)
print("runs folder:", RUNS_ROOT, "contents:", os.listdir(RUNS_ROOT))

## 2. Get / update the code
If the repository is private, add a Colab secret named `GITHUB_TOKEN` (key icon on the left) with a GitHub token that can read the repo.

In [ ]:
import os, subprocess
url = REPO
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    if tok:
        url = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists('/content/TIDE_GR'):
    subprocess.run(['git', 'clone', '-q', url, '/content/TIDE_GR'], check=True)
else:
    subprocess.run(['git', '-C', '/content/TIDE_GR', 'pull', '-q'], check=True)
%cd /content/TIDE_GR
!git log -1 --oneline
!pip install -q -r requirements.txt

## 3. Quick check (≈15–30 min, run once)
Builds the dataset (cached in the shared folder), runs TIDE-GR / LightGCN-FT / SASRec on a short stream,
checks the metrics and the resume logic, and **prints the estimated hours of the full plan**.

In [ ]:
!python -m tidegr.quick_check --data data/amazon_vg.yaml --runs_root "$RUNS_ROOT"

## 4. Full experiment plan (resumable — just run this cell again after any interruption)
Order: tuning on history only → TIDE-GR + baselines (seed 0) → more seeds → ablations → optional runs.
Run only some runs with `--only tidegr,lightgcn_ft`.

In [ ]:
!python -m tidegr.run_all --plan plans/amazon_vg.yaml --runs_root "$RUNS_ROOT" --max_hours $MAX_HOURS

## 5. Progress

In [ ]:
import json
st = json.load(open(f"{RUNS_ROOT}/amazon_vg/status.json"))
for k, v in st.items():
    print(f"{k:35s} {v.get('state'):8s} {v.get('ndcg@20') or ''}")

## 6. Results → Drive + Google Sheet + paper figures (run any time; re-run after new runs finish)
* Tables: `<RUNS_ROOT>/amazon_vg/results/` (summary.csv, table.md, significance.csv)
* Figures (PDF for LaTeX + PNG): `<RUNS_ROOT>/amazon_vg/results/figures/`
* Google Sheet tabs: `amazon_vg_table`, `amazon_vg_runs`, `amazon_vg_blocks`, `amazon_vg_significance`, `amazon_vg_status`, `log`
The first time, Google asks you to allow access (Sheets + Drive). Each run re-writes the tabs from the files on Drive.

In [ ]:
!python -m tidegr.report --runs_root "$RUNS_ROOT" --dataset amazon_vg

In [ ]:
import sys; sys.path.insert(0, '/content/TIDE_GR')
from tidegr.sheets import sync
sync(RUNS_ROOT, 'amazon_vg', SHEET, sa_key=SA_KEY)

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(f'{RUNS_ROOT}/amazon_vg/results/figures/*.png')):
    print(p); display(Image(p, width=520))